# Conservative Switch v1 Experiment

**Purpose.** Test one focused response to the loss-taxonomy result: retreat
only when the active Pokémon is not ready and a benched Pokémon is ready.

**Single intended change.** Keep the promoted development-first ordering,
except prioritize a legal `RETREAT` in that specific ready-bench situation
and choose the best ready benched target afterward.

**Promotion gate.** The candidate must clear promoted, planner v2, random,
and pressure controls without runtime failures. Production remains frozen
unless the gate clears.

## 1. Configuration and frozen sources

The candidate and all non-random controls are embedded by the deterministic
local builder. The official random policy and simulator are loaded from the
Kaggle competition input.

In [ ]:
from collections import Counter
from copy import deepcopy
from pathlib import Path
import importlib.util
import json
import random
import shutil
import sys
import time

import numpy as np
import pandas as pd

BASE_SEED = 20260626
GAMES_PER_CELL = 10
MAX_DECISIONS = 5_000
BOOTSTRAP_SAMPLES = 10_000
WORK_DIR = Path("/kaggle/working/switch_v1_runtime")
REPLAY_DIR = Path("/kaggle/working/switch_v1_replays")

CANDIDATE_SOURCE = '"""Development-first agent with a conservative ready-bench retreat rule."""\n\nfrom __future__ import annotations\n\nfrom pathlib import Path\n\nfrom cg.api import AreaType, OptionType, SelectContext, SelectType, to_observation_class\n\n\nMAIN_ACTION_PRIORITY = {\n    OptionType.EVOLVE: 0,\n    OptionType.ABILITY: 1,\n    OptionType.ATTACH: 2,\n    OptionType.PLAY: 3,\n    OptionType.ATTACK: 4,\n    OptionType.RETREAT: 5,\n    OptionType.DISCARD: 6,\n    OptionType.END: 7,\n}\n\n\ndef read_deck_csv() -> list[int]:\n    """Load the 60 card IDs from the submission directory."""\n    candidates = (\n        Path(__file__).resolve().with_name("deck.csv"),\n        Path("deck.csv"),\n        Path("/kaggle_simulations/agent/deck.csv"),\n    )\n    path = next((candidate for candidate in candidates if candidate.exists()), None)\n    if path is None:\n        raise FileNotFoundError("Could not locate deck.csv beside the agent or at runtime paths.")\n    deck = [int(line.strip()) for line in path.read_text().splitlines() if line.strip()]\n    if len(deck) != 60:\n        raise ValueError(f"Expected 60 cards in {path}, found {len(deck)}.")\n    return deck\n\n\ndef _stable_key(option: object, index: int) -> tuple[int, ...]:\n    """Build a stable tie-break key without assuming all SDK fields exist."""\n    fields = (\n        "number",\n        "playerIndex",\n        "area",\n        "index",\n        "inPlayArea",\n        "inPlayIndex",\n        "attackId",\n        "cardId",\n        "serial",\n    )\n    values = []\n    for field in fields:\n        value = getattr(option, field, None)\n        try:\n            values.append(int(value) if value is not None else 1_000_000)\n        except (TypeError, ValueError):\n            values.append(1_000_000)\n    return (*values, index)\n\n\ndef _energy_count(pokemon: object | None) -> int:\n    if pokemon is None:\n        return 0\n    energies = getattr(pokemon, "energies", None)\n    if energies is None:\n        energies = getattr(pokemon, "energyCards", [])\n    return len(energies)\n\n\ndef _card_id(pokemon: object | None) -> int:\n    if pokemon is None:\n        return -1\n    try:\n        return int(getattr(pokemon, "id", -1))\n    except (TypeError, ValueError):\n        return -1\n\n\ndef _ready_bench_indices(obs: object) -> set[int]:\n    """Return indices of benched Pokémon with at least one attached Energy."""\n    player = int(obs.current.yourIndex)\n    bench = obs.current.players[player].bench\n    return {\n        index\n        for index, pokemon in enumerate(bench)\n        if pokemon is not None and _energy_count(pokemon) >= 1\n    }\n\n\ndef _should_retreat_to_ready_bench(obs: object) -> bool:\n    """Retreat only when active is unready and a ready bench attacker exists."""\n    player = int(obs.current.yourIndex)\n    active = obs.current.players[player].active\n    active_energy = _energy_count(active[0] if active else None)\n    if active_energy > 0:\n        return False\n    if not _ready_bench_indices(obs):\n        return False\n    return any(option.type == OptionType.RETREAT for option in obs.select.option)\n\n\ndef _bench_target_score(obs: object, option: object, index: int) -> tuple[int, tuple[int, ...]]:\n    player = int(obs.current.yourIndex)\n    if int(getattr(option, "playerIndex", player)) != player:\n        return (-10_000, _stable_key(option, index))\n    if getattr(option, "area", None) != AreaType.BENCH:\n        return (-10_000, _stable_key(option, index))\n    bench_index = int(getattr(option, "index", -1))\n    if bench_index not in _ready_bench_indices(obs):\n        return (-10_000, _stable_key(option, index))\n    pokemon = obs.current.players[player].bench[bench_index]\n    card_id = _card_id(pokemon)\n    energy = _energy_count(pokemon)\n    score = 10_000 + 1_000 * energy\n    if card_id == 723:\n        score += 500\n    elif card_id == 721:\n        score += 300\n    elif card_id == 722:\n        score += 100\n    return (score, _stable_key(option, index))\n\n\ndef _choose_indices(obs: object) -> list[int]:\n    """Return deterministic legal indices for the current selection."""\n    select = obs.select\n    options = list(select.option)\n    if not options:\n        return []\n\n    indexed = list(enumerate(options))\n    context = getattr(select, "context", None)\n\n    if context == SelectContext.MULLIGAN:\n        no_choices = [pair for pair in indexed if pair[1].type == OptionType.NO]\n        if no_choices:\n            indexed = no_choices\n    elif getattr(select, "type", None) == SelectType.MAIN:\n        if _should_retreat_to_ready_bench(obs):\n            retreat_choices = [pair for pair in indexed if pair[1].type == OptionType.RETREAT]\n            indexed = retreat_choices or indexed\n        indexed.sort(\n            key=lambda pair: (\n                0 if pair[1].type == OptionType.RETREAT and _should_retreat_to_ready_bench(obs)\n                else MAIN_ACTION_PRIORITY.get(pair[1].type, 99),\n                _stable_key(pair[1], pair[0]),\n            )\n        )\n    elif context in (SelectContext.SWITCH, SelectContext.TO_ACTIVE):\n        indexed.sort(key=lambda pair: (-_bench_target_score(obs, pair[1], pair[0])[0], _stable_key(pair[1], pair[0])))\n    else:\n        indexed.sort(key=lambda pair: _stable_key(pair[1], pair[0]))\n\n    required = max(0, int(select.minCount))\n    requested = required if required > 0 else min(1, int(select.maxCount))\n    count = min(requested, int(select.maxCount), len(indexed))\n    chosen = [index for index, _ in indexed[:count]]\n\n    if not (select.minCount <= len(chosen) <= select.maxCount):\n        raise ValueError("Policy produced an invalid selection count.")\n    if len(chosen) != len(set(chosen)):\n        raise ValueError("Policy produced duplicate option indices.")\n    return chosen\n\n\ndef agent(obs_dict: dict) -> list[int]:\n    """Return a legal deck or deterministic action for an observation."""\n    obs = to_observation_class(obs_dict)\n    if obs.select is None:\n        return read_deck_csv()\n    return _choose_indices(obs)\n'
PROMOTED_SOURCE = '"""Development-first deterministic agent for Pok?mon TCG AI Battle."""\n\nfrom __future__ import annotations\n\nfrom pathlib import Path\n\nfrom cg.api import OptionType, SelectContext, SelectType, to_observation_class\n\n\nMAIN_ACTION_PRIORITY = {\n    OptionType.EVOLVE: 0,\n    OptionType.ABILITY: 1,\n    OptionType.ATTACH: 2,\n    OptionType.PLAY: 3,\n    OptionType.ATTACK: 4,\n    OptionType.RETREAT: 5,\n    OptionType.DISCARD: 6,\n    OptionType.END: 7,\n}\n\n\ndef read_deck_csv() -> list[int]:\n    """Load the 60 card IDs from the submission directory."""\n    candidates = (\n        Path(__file__).resolve().with_name("deck.csv"),\n        Path("deck.csv"),\n        Path("/kaggle_simulations/agent/deck.csv"),\n    )\n    path = next((candidate for candidate in candidates if candidate.exists()), None)\n    if path is None:\n        raise FileNotFoundError("Could not locate deck.csv beside the agent or at runtime paths.")\n    deck = [int(line.strip()) for line in path.read_text().splitlines() if line.strip()]\n    if len(deck) != 60:\n        raise ValueError(f"Expected 60 cards in {path}, found {len(deck)}.")\n    return deck\n\n\ndef _stable_key(option: object, index: int) -> tuple[int, ...]:\n    """Build a stable tie-break key without assuming all SDK fields exist."""\n    fields = (\n        "number",\n        "playerIndex",\n        "area",\n        "index",\n        "inPlayArea",\n        "inPlayIndex",\n        "attackId",\n        "cardId",\n        "serial",\n    )\n    values = []\n    for field in fields:\n        value = getattr(option, field, None)\n        try:\n            values.append(int(value) if value is not None else 1_000_000)\n        except (TypeError, ValueError):\n            values.append(1_000_000)\n    return (*values, index)\n\n\ndef _choose_indices(obs: object) -> list[int]:\n    """Return deterministic legal indices for the current selection."""\n    select = obs.select\n    options = list(select.option)\n    if not options:\n        return []\n\n    indexed = list(enumerate(options))\n    context = getattr(select, "context", None)\n\n    if context == SelectContext.MULLIGAN:\n        no_choices = [pair for pair in indexed if pair[1].type == OptionType.NO]\n        if no_choices:\n            indexed = no_choices\n    elif getattr(select, "type", None) == SelectType.MAIN:\n        indexed.sort(\n            key=lambda pair: (\n                MAIN_ACTION_PRIORITY.get(pair[1].type, 99),\n                _stable_key(pair[1], pair[0]),\n            )\n        )\n    else:\n        indexed.sort(key=lambda pair: _stable_key(pair[1], pair[0]))\n\n    required = max(0, int(select.minCount))\n    requested = required if required > 0 else min(1, int(select.maxCount))\n    count = min(requested, int(select.maxCount), len(indexed))\n    chosen = [index for index, _ in indexed[:count]]\n\n    if not (select.minCount <= len(chosen) <= select.maxCount):\n        raise ValueError("Policy produced an invalid selection count.")\n    if len(chosen) != len(set(chosen)):\n        raise ValueError("Policy produced duplicate option indices.")\n    return chosen\n\n\ndef agent(obs_dict: dict) -> list[int]:\n    """Return a legal deck or deterministic action for an observation."""\n    obs = to_observation_class(obs_dict)\n    if obs.select is None:\n        return read_deck_csv()\n    return _choose_indices(obs)\n'
PLANNER_V2_SOURCE = '"""Stateless attack-planner candidate for the Mega Abomasnow/Kyogre deck."""\n\nfrom __future__ import annotations\n\nfrom collections import Counter\nfrom dataclasses import dataclass\nfrom pathlib import Path\n\nfrom cg.api import AreaType, OptionType, SelectContext, SelectType, to_observation_class\n\n\nKYOGRE = 721\nSNOVER = 722\nMEGA_ABOMASNOW_EX = 723\nWATER_ENERGY = 3\n\nRIPTIDE = 1042\nHAMMER_LANCHE = 1046\n\nDECK_WATER_ENERGY = 35\n\nMAIN_ACTION_PRIORITY = {\n    OptionType.EVOLVE: 0,\n    OptionType.ABILITY: 1,\n    OptionType.ATTACH: 2,\n    OptionType.PLAY: 3,\n    OptionType.ATTACK: 4,\n    OptionType.RETREAT: 5,\n    OptionType.DISCARD: 6,\n    OptionType.END: 7,\n}\n\n\n@dataclass(frozen=True)\nclass AttackPlan:\n    """A plan reconstructed from the current public observation."""\n\n    attacker_area: object | None = None\n    attacker_index: int = -1\n    attacker_id: int = -1\n    attack_id: int = -1\n    expected_damage: float = 0.0\n    target_hp: int = 0\n    energy_needed: int = 0\n    ready: bool = False\n    immediate_ko: bool = False\n    confident: bool = False\n\n\ndef read_deck_csv() -> list[int]:\n    """Load and validate the candidate\'s frozen 60-card deck."""\n    candidates = (\n        Path(__file__).resolve().with_name("deck.csv"),\n        Path("deck.csv"),\n        Path("/kaggle_simulations/agent/deck.csv"),\n    )\n    path = next((candidate for candidate in candidates if candidate.exists()), None)\n    if path is None:\n        raise FileNotFoundError("Could not locate deck.csv.")\n    deck = [int(line.strip()) for line in path.read_text().splitlines() if line.strip()]\n    if len(deck) != 60:\n        raise ValueError(f"Expected 60 cards in {path}, found {len(deck)}.")\n    return deck\n\n\ndef _stable_key(option: object, index: int) -> tuple[int, ...]:\n    fields = (\n        "number", "playerIndex", "area", "index", "inPlayArea",\n        "inPlayIndex", "attackId", "cardId", "serial",\n    )\n    values = []\n    for field in fields:\n        value = getattr(option, field, None)\n        try:\n            values.append(int(value) if value is not None else 1_000_000)\n        except (TypeError, ValueError):\n            values.append(1_000_000)\n    return (*values, index)\n\n\ndef _get_card(obs: object, area: object, index: int, player: int):\n    """Resolve a visible card without assuming that every zone is populated."""\n    state = obs.current\n    player_state = state.players[player]\n    zones = {\n        AreaType.HAND: player_state.hand,\n        AreaType.DISCARD: player_state.discard,\n        AreaType.ACTIVE: player_state.active,\n        AreaType.BENCH: player_state.bench,\n        AreaType.PRIZE: player_state.prize,\n        AreaType.STADIUM: state.stadium,\n        AreaType.LOOKING: state.looking,\n    }\n    if area == AreaType.DECK:\n        zone = getattr(obs.select, "deck", [])\n    else:\n        zone = zones.get(area, [])\n    if index is None or not 0 <= int(index) < len(zone):\n        return None\n    return zone[int(index)]\n\n\ndef _energy_count(pokemon: object | None) -> int:\n    if pokemon is None:\n        return 0\n    return len(getattr(pokemon, "energies", []))\n\n\ndef _visible_counts(obs: object, player: int) -> tuple[Counter, Counter, Counter]:\n    player_state = obs.current.players[player]\n    field = Counter()\n    hand = Counter()\n    discard = Counter()\n    for card in list(player_state.active) + list(player_state.bench):\n        if card is not None:\n            field[int(card.id)] += 1\n    for card in player_state.hand:\n        hand[int(card.id)] += 1\n    for card in player_state.discard:\n        discard[int(card.id)] += 1\n    return field, hand, discard\n\n\ndef _opponent_active_hp(obs: object, player: int) -> int:\n    active = obs.current.players[1 - player].active\n    if not active or active[0] is None:\n        return 0\n    return int(active[0].hp)\n\n\ndef _expected_hammer_lanche(obs: object, player: int) -> float:\n    """Estimate six-card Energy hits using only the candidate\'s visible cards."""\n    player_state = obs.current.players[player]\n    visible_water = 0\n    for card in player_state.hand:\n        visible_water += int(card.id == WATER_ENERGY)\n    for card in player_state.discard:\n        visible_water += int(card.id == WATER_ENERGY)\n    for pokemon in list(player_state.active) + list(player_state.bench):\n        if pokemon is None:\n            continue\n        for energy in getattr(pokemon, "energyCards", []):\n            visible_water += int(energy.id == WATER_ENERGY)\n    remaining_water = max(0, DECK_WATER_ENERGY - visible_water)\n    deck_count = max(1, int(player_state.deckCount))\n    return 600.0 * min(1.0, remaining_water / deck_count)\n\n\ndef _attack_damage(card_id: int, discard_water: int, hammer_estimate: float) -> tuple[int, float]:\n    if card_id == KYOGRE:\n        return RIPTIDE, float(discard_water * 20)\n    if card_id == MEGA_ABOMASNOW_EX:\n        return HAMMER_LANCHE, hammer_estimate\n    return -1, 0.0\n\n\ndef _build_plan(obs: object) -> AttackPlan:\n    """Choose one attacker using readiness, expected damage, and switch cost."""\n    state = obs.current\n    player = int(state.yourIndex)\n    player_state = state.players[player]\n    _, _, discard = _visible_counts(obs, player)\n    target_hp = _opponent_active_hp(obs, player)\n    if target_hp <= 0:\n        return AttackPlan()\n\n    hammer_estimate = _expected_hammer_lanche(obs, player)\n    candidates = []\n    for area, cards in ((AreaType.ACTIVE, player_state.active), (AreaType.BENCH, player_state.bench)):\n        for index, card in enumerate(cards):\n            if card is None or int(card.id) not in (KYOGRE, MEGA_ABOMASNOW_EX):\n                continue\n            required = 1 if int(card.id) == KYOGRE else 2\n            attached = _energy_count(card)\n            needed = max(0, required - attached)\n            attack_id, damage = _attack_damage(\n                int(card.id), discard[WATER_ENERGY], hammer_estimate\n            )\n            if int(card.id) == KYOGRE and damage <= 0:\n                continue\n            ready = needed == 0\n            immediate_ko = ready and damage >= target_hp\n            score = damage\n            score += 10_000 if immediate_ko else 0\n            score += 600 if ready else -350 * needed\n            score += 150 if area == AreaType.ACTIVE else -100\n            score += 40 if int(card.id) == MEGA_ABOMASNOW_EX else 0\n            candidates.append((score, area, index, card, attack_id, damage, needed, ready, immediate_ko))\n\n    if not candidates:\n        return AttackPlan()\n    best = max(candidates, key=lambda item: (item[0], -item[2]))\n    _, area, index, card, attack_id, damage, needed, ready, immediate_ko = best\n    return AttackPlan(\n        attacker_area=area,\n        attacker_index=index,\n        attacker_id=int(card.id),\n        attack_id=attack_id,\n        expected_damage=damage,\n        target_hp=target_hp,\n        energy_needed=needed,\n        ready=ready,\n        immediate_ko=immediate_ko,\n        confident=True,\n    )\n\n\ndef _attachment_score(obs: object, option: object, plan: AttackPlan) -> int:\n    player = int(obs.current.yourIndex)\n    pokemon = _get_card(obs, option.inPlayArea, option.inPlayIndex, player)\n    if pokemon is None:\n        return -10_000\n    card_id = int(pokemon.id)\n    energy = _energy_count(pokemon)\n    required = 1 if card_id == KYOGRE else 2 if card_id in (SNOVER, MEGA_ABOMASNOW_EX) else 99\n    score = 0\n    if (\n        plan.confident\n        and plan.energy_needed > 0\n        and option.inPlayArea == plan.attacker_area\n        and option.inPlayIndex == plan.attacker_index\n    ):\n        score += 4_000\n    if energy < required:\n        score += 1_000 - 100 * energy\n    else:\n        score -= 2_000 + 200 * (energy - required)\n    if card_id == MEGA_ABOMASNOW_EX:\n        score += 250\n    elif card_id == SNOVER:\n        score += 150\n    elif card_id == KYOGRE:\n        score += 100\n    if option.inPlayArea == AreaType.ACTIVE:\n        score += 40\n    return score\n\n\ndef _main_score(obs: object, option: object, plan: AttackPlan) -> int:\n    player = int(obs.current.yourIndex)\n    base = 8_000 - 1_000 * MAIN_ACTION_PRIORITY.get(option.type, 8)\n    if option.type == OptionType.EVOLVE:\n        pokemon = _get_card(obs, option.inPlayArea, option.inPlayIndex, player)\n        if pokemon is not None and int(pokemon.id) == SNOVER:\n            base += 2_000\n            if option.inPlayArea == plan.attacker_area and option.inPlayIndex == plan.attacker_index:\n                base += 1_000\n    elif option.type == OptionType.ATTACH:\n        base += _attachment_score(obs, option, plan)\n    elif option.type == OptionType.RETREAT:\n        if plan.confident and plan.attacker_area == AreaType.BENCH and plan.ready:\n            base += 5_000\n        else:\n            base -= 5_000\n    elif option.type == OptionType.ATTACK:\n        if plan.confident and plan.attacker_area == AreaType.ACTIVE:\n            if int(getattr(option, "attackId", -1)) == plan.attack_id:\n                base += 2_000\n                if plan.immediate_ko:\n                    base += 8_000\n        else:\n            base -= 2_000\n    elif option.type == OptionType.PLAY:\n        card = _get_card(obs, AreaType.HAND, option.index, player)\n        if card is not None and int(card.id) == 1227:\n            field, hand, _ = _visible_counts(obs, player)\n            if field[SNOVER] and hand[MEGA_ABOMASNOW_EX]:\n                base -= 4_000\n    return base\n\n\ndef _card_score(obs: object, option: object, plan: AttackPlan) -> int:\n    player = int(obs.current.yourIndex)\n    card = _get_card(obs, option.area, option.index, int(option.playerIndex))\n    if card is None:\n        return -10_000\n    card_id = int(card.id)\n    context = obs.select.context\n    energy = _energy_count(card)\n    field, hand, _ = _visible_counts(obs, player)\n\n    if context in (SelectContext.SWITCH, SelectContext.TO_ACTIVE):\n        score = energy * 100\n        if int(option.playerIndex) == player:\n            if plan.confident and option.area == AreaType.BENCH and option.index == plan.attacker_index:\n                score += 10_000\n            score += 500 if card_id == MEGA_ABOMASNOW_EX else 300 if card_id == KYOGRE else 0\n        return score\n    if context == SelectContext.SETUP_ACTIVE_POKEMON:\n        going_first = int(obs.current.firstPlayer) == player\n        if going_first:\n            return 500 if card_id == SNOVER else 350 if card_id == KYOGRE else 0\n        return 500 if card_id == KYOGRE else 400 if card_id == SNOVER else 0\n    if context in (SelectContext.TO_HAND, SelectContext.TO_BENCH):\n        if card_id == MEGA_ABOMASNOW_EX:\n            return 1_000 if field[SNOVER] and not hand[MEGA_ABOMASNOW_EX] else 250\n        if card_id == SNOVER:\n            return 800 if not field[SNOVER] else 300\n        if card_id == KYOGRE:\n            return 600 if not field[KYOGRE] else 150\n    if context == SelectContext.DISCARD:\n        score = 1_000 if card_id == WATER_ENERGY else 0\n        score += 500 if hand[card_id] >= 2 else 0\n        if card_id == MEGA_ABOMASNOW_EX and field[SNOVER]:\n            score -= 1_000\n        return score\n    if context == SelectContext.ATTACH_FROM:\n        pseudo = type("AttachTarget", (), {\n            "inPlayArea": option.area,\n            "inPlayIndex": option.index,\n        })()\n        return _attachment_score(obs, pseudo, plan)\n    return energy * 10\n\n\ndef _fallback_pairs(obs: object) -> list[tuple[int, object]]:\n    indexed = list(enumerate(obs.select.option))\n    if obs.select.context == SelectContext.MULLIGAN:\n        no_choices = [pair for pair in indexed if pair[1].type == OptionType.NO]\n        if no_choices:\n            return no_choices\n    if obs.select.type == SelectType.MAIN:\n        return sorted(indexed, key=lambda pair: (\n            MAIN_ACTION_PRIORITY.get(pair[1].type, 99), _stable_key(pair[1], pair[0])\n        ))\n    return sorted(indexed, key=lambda pair: _stable_key(pair[1], pair[0]))\n\n\ndef _choose_indices(obs: object) -> list[int]:\n    select = obs.select\n    if not select.option:\n        return []\n    fallback = _fallback_pairs(obs)\n    plan = _build_plan(obs)\n\n    scored = []\n    for fallback_rank, (index, option) in enumerate(fallback):\n        score = -fallback_rank\n        if select.type == SelectType.MAIN and plan.confident:\n            score += _main_score(obs, option, plan) * 100\n        elif option.type == OptionType.CARD:\n            score += _card_score(obs, option, plan) * 100\n        elif option.type == OptionType.YES and select.context == SelectContext.IS_FIRST:\n            score += 10_000\n        elif option.type == OptionType.NUMBER:\n            score += int(option.number) * 100\n        scored.append((score, _stable_key(option, index), index))\n\n    scored.sort(key=lambda item: (-item[0], item[1]))\n    required = max(0, int(select.minCount))\n    requested = required if required > 0 else min(1, int(select.maxCount))\n    count = min(requested, int(select.maxCount), len(scored))\n    chosen = [index for _, _, index in scored[:count]]\n    if not (select.minCount <= len(chosen) <= select.maxCount):\n        raise ValueError("Planner produced an invalid selection count.")\n    if len(chosen) != len(set(chosen)):\n        raise ValueError("Planner produced duplicate indices.")\n    return chosen\n\n\ndef agent(obs_dict: dict) -> list[int]:\n    """Return the frozen deck or a legal stateless planner action."""\n    obs = to_observation_class(obs_dict)\n    if obs.select is None:\n        return read_deck_csv()\n    return _choose_indices(obs)\n'
PRESSURE_SOURCE = '"""Frozen attack-priority pressure control for evaluation only."""\n\nfrom __future__ import annotations\n\nfrom pathlib import Path\n\nfrom cg.api import OptionType, SelectContext, SelectType, to_observation_class\n\nMAIN_ACTION_PRIORITY = {\n    OptionType.ATTACK: 0,\n    OptionType.EVOLVE: 1,\n    OptionType.ATTACH: 2,\n    OptionType.ABILITY: 3,\n    OptionType.PLAY: 4,\n    OptionType.RETREAT: 5,\n    OptionType.DISCARD: 6,\n    OptionType.END: 7,\n}\n\n\ndef read_deck_csv() -> list[int]:\n    candidates = (\n        Path(__file__).resolve().with_name("deck.csv"),\n        Path("deck.csv"),\n        Path("/kaggle_simulations/agent/deck.csv"),\n    )\n    path = next((candidate for candidate in candidates if candidate.exists()), None)\n    if path is None:\n        raise FileNotFoundError("Could not locate deck.csv.")\n    deck = [int(line.strip()) for line in path.read_text().splitlines() if line.strip()]\n    if len(deck) != 60:\n        raise ValueError(f"Expected 60 cards in {path}, found {len(deck)}.")\n    return deck\n\n\ndef _stable_key(option: object, index: int) -> tuple[int, ...]:\n    fields = ("number", "playerIndex", "area", "index", "inPlayArea", "inPlayIndex", "attackId", "cardId", "serial")\n    values = []\n    for field in fields:\n        value = getattr(option, field, None)\n        try:\n            values.append(int(value) if value is not None else 1_000_000)\n        except (TypeError, ValueError):\n            values.append(1_000_000)\n    return (*values, index)\n\n\ndef _choose_indices(obs: object) -> list[int]:\n    select = obs.select\n    indexed = list(enumerate(select.option))\n    if not indexed:\n        return []\n    if select.context == SelectContext.MULLIGAN:\n        no_choices = [pair for pair in indexed if pair[1].type == OptionType.NO]\n        if no_choices:\n            indexed = no_choices\n    elif select.context == SelectContext.IS_FIRST:\n        yes_choices = [pair for pair in indexed if pair[1].type == OptionType.YES]\n        if yes_choices:\n            indexed = yes_choices\n    elif select.type == SelectType.MAIN:\n        indexed.sort(key=lambda pair: (MAIN_ACTION_PRIORITY.get(pair[1].type, 99), _stable_key(pair[1], pair[0])))\n    else:\n        indexed.sort(key=lambda pair: _stable_key(pair[1], pair[0]))\n    required = max(0, int(select.minCount))\n    requested = required if required > 0 else min(1, int(select.maxCount))\n    count = min(requested, int(select.maxCount), len(indexed))\n    chosen = [index for index, _ in indexed[:count]]\n    if not (select.minCount <= len(chosen) <= select.maxCount):\n        raise ValueError("Policy produced an invalid selection count.")\n    if len(chosen) != len(set(chosen)):\n        raise ValueError("Policy produced duplicate option indices.")\n    return chosen\n\n\ndef agent(obs_dict: dict) -> list[int]:\n    obs = to_observation_class(obs_dict)\n    if obs.select is None:\n        return read_deck_csv()\n    return _choose_indices(obs)\n'
DECK_SOURCE = '1158\n721\n721\n722\n722\n722\n722\n723\n723\n723\n723\n1145\n1145\n1145\n1145\n1205\n1205\n1227\n1227\n1227\n1227\n1235\n1235\n1235\n1235\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n3\n'

def first_match(pattern: str) -> Path:
    matches = sorted(Path("/kaggle/input").rglob(pattern))
    if not matches:
        raise FileNotFoundError(f"No Kaggle input matched {pattern}")
    return matches[0]

def load_module(name: str, path: Path):
    spec = importlib.util.spec_from_file_location(name, path)
    module = importlib.util.module_from_spec(spec)
    sys.modules[name] = module
    spec.loader.exec_module(module)
    return module

sample_dir = first_match("sample_submission/main.py").parent
print(f"Official random source: {sample_dir / 'main.py'}")

## 2. Runtime staging

Stage each policy in one official simulator copy. All policies must share
the same frozen 60-card deck before the tournament starts.

In [ ]:
if WORK_DIR.exists():
    shutil.rmtree(WORK_DIR)
shutil.copytree(sample_dir, WORK_DIR)
REPLAY_DIR.mkdir(parents=True, exist_ok=True)

(WORK_DIR / "deck.csv").write_text(DECK_SOURCE, encoding="utf-8")
(WORK_DIR / "candidate_main.py").write_text(CANDIDATE_SOURCE, encoding="utf-8")
(WORK_DIR / "promoted_main.py").write_text(PROMOTED_SOURCE, encoding="utf-8")
(WORK_DIR / "planner_v2_main.py").write_text(PLANNER_V2_SOURCE, encoding="utf-8")
(WORK_DIR / "pressure_main.py").write_text(PRESSURE_SOURCE, encoding="utf-8")

sys.path.insert(0, str(WORK_DIR))
from cg.api import OptionType, SelectContext, SelectType, to_observation_class
from cg.game import battle_finish, battle_select, battle_start, visualize_data

candidate = load_module("conservative_switch_v1", WORK_DIR / "candidate_main.py")
promoted = load_module("promoted_frozen", WORK_DIR / "promoted_main.py")
planner_v2 = load_module("planner_v2_frozen", WORK_DIR / "planner_v2_main.py")
pressure = load_module("pressure_control_frozen", WORK_DIR / "pressure_main.py")
random_control = load_module("official_random_control", sample_dir / "main.py")

deck = candidate.read_deck_csv()
assert deck == promoted.read_deck_csv() == planner_v2.read_deck_csv() == pressure.read_deck_csv()
assert len(deck) == 60
print("PASS: candidate and controls share the frozen 60-card deck")

## 3. Controlled play and switch telemetry

The wrapper controls only first-player choice. Candidate main decisions log
whether retreat was available and whether it was chosen.

In [ ]:
def enum_name(enum_class, value) -> str:
    try:
        return enum_class(value).name
    except (ValueError, TypeError):
        return f"UNKNOWN_{value}"

def validate_action(obs, action: list[int]) -> None:
    select = obs.select
    assert isinstance(action, list)
    assert all(isinstance(index, int) for index in action)
    assert len(action) == len(set(action))
    assert select.minCount <= len(action) <= select.maxCount
    assert all(0 <= index < len(select.option) for index in action)

class ForcedFirstChoice:
    def __init__(self, base_policy, player_zero_goes_first: bool):
        self.base_policy = base_policy
        self.player_zero_goes_first = player_zero_goes_first

    def agent(self, obs_dict: dict) -> list[int]:
        obs = to_observation_class(obs_dict)
        if obs.select is not None and obs.select.context == SelectContext.IS_FIRST:
            desired = OptionType.YES if self.player_zero_goes_first else OptionType.NO
            matches = [i for i, option in enumerate(obs.select.option) if option.type == desired]
            if not matches:
                raise ValueError(f"IS_FIRST does not expose {desired.name}")
            return [matches[0]]
        return self.base_policy.agent(obs_dict)

def energy_count(pokemon) -> int:
    if pokemon is None:
        return 0
    energies = getattr(pokemon, "energies", None)
    if energies is None:
        energies = getattr(pokemon, "energyCards", [])
    return len(energies)

def switch_features(obs, chosen_names: list[str]) -> dict:
    player = int(obs.current.yourIndex)
    player_state = obs.current.players[player]
    active = player_state.active[0] if player_state.active else None
    bench = [card for card in player_state.bench if card is not None]
    available = {enum_name(OptionType, option.type) for option in obs.select.option}
    return {
        "active_energy": energy_count(active),
        "ready_bench": sum(energy_count(card) >= 1 for card in bench),
        "retreat_available": "RETREAT" in available,
        "attack_available": "ATTACK" in available,
        "chosen_action": chosen_names[0] if chosen_names else "SKIP",
    }

replay_counts = Counter()
replay_paths = []

def replay_observation(obs_dict: dict) -> dict:
    saved = deepcopy(obs_dict)
    saved.pop("search_begin_input", None)
    return saved

def write_replay(obs_log: list, action_log: list, path: Path) -> None:
    visual = json.loads(visualize_data())
    for index, step in enumerate(visual):
        if index < len(obs_log):
            step["obs"] = obs_log[index]
        if index < len(action_log):
            step["action"] = [action_log[index], action_log[index]]
    path.write_text(json.dumps(visual), encoding="utf-8")

In [ ]:
def play_game(opponent_policy, opponent_name: str, game_id: int, candidate_player: int, candidate_should_go_first: bool):
    random.seed(BASE_SEED + game_id)
    started = time.perf_counter()
    decisions, first_player = 0, None
    candidate_actions = Counter()
    snapshots = []
    obs_log, action_log = [""], [None]
    player_zero_first = candidate_should_go_first if candidate_player == 0 else not candidate_should_go_first
    policies = (
        {0: ForcedFirstChoice(candidate, player_zero_first), 1: opponent_policy}
        if candidate_player == 0
        else {0: ForcedFirstChoice(opponent_policy, player_zero_first), 1: candidate}
    )
    matchup = f"switch_v1_vs_{opponent_name}"
    try:
        obs_dict, start_data = battle_start(deck, deck)
        if obs_dict is None:
            return ({"status": "start_error", "matchup": matchup, "game": game_id, "candidate_player": candidate_player, "error_player": start_data.errorPlayer, "error_type": start_data.errorType}, snapshots)

        while decisions < MAX_DECISIONS:
            obs = to_observation_class(obs_dict)
            observed_first = getattr(obs.current, "firstPlayer", None)
            if observed_first in (0, 1):
                first_player = int(observed_first)
            if obs.current.result != -1:
                winner = int(obs.current.result)
                score = 1.0 if winner == candidate_player else 0.0 if winner in (0, 1) else 0.5
                replay_path = None
                key = (matchup, candidate_player, first_player == candidate_player, score)
                if score == 0.0 and replay_counts[key] < 1:
                    replay_path = REPLAY_DIR / f"{matchup}_game_{game_id}_seat_{candidate_player}_first_{first_player}.json"
                    write_replay(obs_log, action_log, replay_path)
                    replay_counts[key] += 1
                    replay_paths.append(str(replay_path))
                return ({
                    "status": "finished", "matchup": matchup, "game": game_id,
                    "candidate_player": candidate_player, "first_player": first_player,
                    "candidate_went_first": first_player == candidate_player,
                    "forced_candidate_went_first": candidate_should_go_first,
                    "winner": winner, "candidate_score": score,
                    "turn": int(obs.current.turn), "decisions": decisions,
                    "seconds": time.perf_counter() - started,
                    "candidate_actions": dict(candidate_actions),
                    "replay_path": str(replay_path) if replay_path else None,
                }, snapshots)

            player = int(obs.current.yourIndex)
            action = policies[player].agent(obs_dict)
            validate_action(obs, action)
            chosen_names = [enum_name(OptionType, obs.select.option[index].type) for index in action]
            if player == candidate_player:
                candidate_actions.update(chosen_names)
                if obs.select.type == SelectType.MAIN:
                    features = switch_features(obs, chosen_names)
                    features.update({
                        "matchup": matchup,
                        "game": game_id,
                        "candidate_player": candidate_player,
                        "candidate_went_first": first_player == candidate_player,
                        "turn": int(obs.current.turn),
                    })
                    snapshots.append(features)
            obs_log.append(replay_observation(obs_dict))
            action_log.append(list(action))
            obs_dict = battle_select(action)
            decisions += 1

        return ({"status": "decision_limit", "matchup": matchup, "game": game_id, "candidate_player": candidate_player, "first_player": first_player, "decisions": decisions}, snapshots)
    except Exception as error:
        return ({"status": "exception", "matchup": matchup, "game": game_id, "candidate_player": candidate_player, "first_player": first_player, "error": f"{type(error).__name__}: {error}", "decisions": decisions}, snapshots)
    finally:
        try:
            battle_finish()
        except Exception:
            pass

## 4. Balanced four-opponent tournament

Each opponent uses ten games in every candidate-seat by forced-turn-order
cell.

In [ ]:
opponents = [
    ("promoted", promoted),
    ("planner_v2", planner_v2),
    ("random", random_control),
    ("pressure", pressure),
]
results, snapshots = [], []
game_id = 120_000
for opponent_name, opponent_policy in opponents:
    for candidate_player in (0, 1):
        for candidate_should_go_first in (False, True):
            for repetition in range(GAMES_PER_CELL):
                result, game_snapshots = play_game(opponent_policy, opponent_name, game_id, candidate_player, candidate_should_go_first)
                results.append(result)
                snapshots.extend(game_snapshots)
                game_id += 1

results_df = pd.DataFrame(results)
snapshots_df = pd.DataFrame(snapshots)
failures = results_df[results_df.status != "finished"]
finished = results_df[results_df.status == "finished"].copy()
assert failures.empty, failures.to_dict("records")
assert (finished.candidate_went_first == finished.forced_candidate_went_first).all()
cells = finished.groupby(["matchup", "candidate_player", "candidate_went_first"]).size()
assert (cells == GAMES_PER_CELL).all(), cells.to_dict()
display(finished.drop(columns=["candidate_actions"], errors="ignore"))
print(f"Completed {len(finished)} games with {len(failures)} failures.")

## 5. Outcome uncertainty and decision

Promote only if the primary promoted-control interval and regression
intervals are all above parity with zero failures.

In [ ]:
rng = np.random.default_rng(BASE_SEED)
summaries = []
for matchup, group in finished.groupby("matchup"):
    scores = group.candidate_score.to_numpy(dtype=float)
    boot = rng.choice(scores, size=(BOOTSTRAP_SAMPLES, len(scores)), replace=True).mean(axis=1)
    summaries.append({
        "matchup": matchup,
        "games": len(scores),
        "wins": int((scores == 1.0).sum()),
        "draws": int((scores == 0.5).sum()),
        "losses": int((scores == 0.0).sum()),
        "score_rate": float(scores.mean()),
        "ci_low": float(np.quantile(boot, 0.025)),
        "ci_high": float(np.quantile(boot, 0.975)),
    })
summary_df = pd.DataFrame(summaries).set_index("matchup")
display(summary_df)

required = ["switch_v1_vs_promoted", "switch_v1_vs_planner_v2", "switch_v1_vs_random", "switch_v1_vs_pressure"]
if len(failures):
    decision = "REJECT: runtime failures"
elif all(summary_df.loc[name].ci_low > 0.5 for name in required):
    decision = "PROMOTE: switch v1 clears all controlled gates"
elif summary_df.loc["switch_v1_vs_promoted"].ci_high < 0.5:
    decision = "REJECT: switch v1 is clearly worse than promoted"
else:
    decision = "HOLD: at least one required interval overlaps parity"
print(f"Decision: {decision}")

## 6. Switch mechanism checks

Confirm that the candidate actually uses retreat and whether that correlates
with controlled-cell score differences.

In [ ]:
attribution = finished.groupby(["matchup", "candidate_player", "candidate_went_first"]).candidate_score.agg(games="size", score_rate="mean")
display(attribution)

mechanism = snapshots_df.groupby("matchup").agg(
    main_decisions=("game", "size"),
    retreat_available_rate=("retreat_available", "mean"),
    retreat_chosen_rate=("chosen_action", lambda s: float((s == "RETREAT").mean())),
    ready_bench_rate=("ready_bench", lambda s: float((s > 0).mean())),
    unready_active_rate=("active_energy", lambda s: float((s == 0).mean())),
    attack_available_rate=("attack_available", "mean"),
    attack_chosen_rate=("chosen_action", lambda s: float((s == "ATTACK").mean())),
)
display(mechanism)

action_rows = []
for result in results:
    for action, count in result.get("candidate_actions", {}).items():
        action_rows.append({"matchup": result["matchup"], "action": action, "count": count})
action_df = pd.DataFrame(action_rows).groupby(["matchup", "action"], as_index=False).sum()
display(action_df.pivot(index="action", columns="matchup", values="count").fillna(0))

## 7. Evidence export

Save aggregate results, per-game records, switch telemetry, and replay
paths. Do not commit replay JSON files.

In [ ]:
payload = {
    "candidate": "conservative_switch_v1",
    "single_change": "retreat when active is unready and a benched attacker is ready",
    "configuration": {
        "base_seed": BASE_SEED,
        "games_per_cell": GAMES_PER_CELL,
        "bootstrap_samples": BOOTSTRAP_SAMPLES,
    },
    "decision": decision,
    "summaries": summary_df.reset_index().to_dict("records"),
    "attribution": attribution.reset_index().to_dict("records"),
    "mechanism": mechanism.reset_index().to_dict("records"),
    "results": results,
    "snapshots": snapshots,
    "replays": replay_paths,
}
evidence_path = Path("/kaggle/working/conservative_switch_experiment.json")
evidence_path.write_text(json.dumps(payload, indent=2, default=str), encoding="utf-8")
print(f"Saved evidence to {evidence_path}")